In [1]:
import pandas as pd
import pickle

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ds.DataFrameOps import DataFrameOperations
from geoai.utils_geo.VectorOps import VectorOperations

raster_ops = RasterOperations()
df_ops = DataFrameOperations()

In [2]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

In [3]:
df_bands = []
array = raster_ops.raster_to_array(RASTER_PATH)
raster_dimension = raster_ops.get_raster_dimensions(RASTER_PATH)


for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
    flat = raster_ops.flatten_array(array, band_index)
    df = df_ops.convert_to_df(flat, band_name)
    df = df.loc[~(df == 0).all(axis=1)]  # remove rows if all of its column is zero
    df_bands.append(df)
final_df_per_bands = pd.concat(df_bands, axis=1)


numerical_columns = final_df_per_bands.select_dtypes(
    include=["float32", "float64"]
).columns.tolist()
scaled_final_df_per_bands = df_ops.scale_to_minmax(final_df_per_bands, numerical_columns)

with open("rf_model.pkl", "rb") as file:
    model = pickle.load(file)

scaled_final_df_per_bands["PREDICTED_LC"] = model.predict(scaled_final_df_per_bands)

Converting raster array to DataFrame with column name BLUE
Converting raster array to DataFrame with column name GREEN
Converting raster array to DataFrame with column name RED
Converting raster array to DataFrame with column name NIR
Converting raster array to DataFrame with column name SWIR


In [4]:
final_df_per_bands

,BLUE,GREEN,RED,NIR,SWIR,PREDICTED_LC
0,0.186575,0.229784,0.242035,0.376729,0.320899,0
1,0.193175,0.223937,0.229691,0.393917,0.323187,0
2,0.192435,0.208770,0.222152,0.309807,0.326178,0
3,0.183112,0.186309,0.190659,0.260887,0.326178,0
4,0.125634,0.132472,0.143119,0.233275,0.276822,0
...,...,...,...,...,...,...
158219,0.128771,0.103759,0.102085,0.326763,0.449347,0
158220,0.135430,0.117305,0.116764,0.336881,0.409669,0
158221,0.134809,0.117305,0.119099,0.331285,0.409669,0
158222,0.084967,0.068968,0.076897,0.272361,0.320195,0


In [5]:
raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC.tif",
    final_df_per_bands,
    "PREDICTED_LC",
    raster_dimension,
    "float32",
)

Raster written to raster_files/PREDICTED_LC.tif
